# Managed VFS replacement probe

Open this notebook in the **launched sandbox** and select **Run in IDE Process**.
Restart the sandbox after building the diagnostic helper. Run cells individually: the dialog step requires interaction.

This uses one real page. It saves a backup under `/tmp`, captures five views of the text, opens the real Replace in Files dialog, and compares the results without saving or refreshing them. The final cell restores the original text. Restoration still leaves journal/undo history of the saves; it does not restore the database's previous journal state.

Native Replace All retains its normal IDE save behaviour. Start with a sandbox whose documents you are ready to save. The probe refuses to start with unsaved edits on the target page.


In [ ]:
%use intellij-platform

import com.intellij.openapi.project.ProjectManager
import com.intellij.util.indexing.IndexableSetContributor
import java.lang.reflect.InvocationTargetException
import java.nio.file.Files

val projectName = "test-project"
val targetPath = "/mywikisource/en/Page/Stevenson-Persuasive Definitions-1938.pdf/13/@wikitext"
val findText = "aspirations"
val replacementText = "aspirations__managed_replace_probe__"
val probeProject = ProjectManager.getInstance().openProjects.single { it.name == projectName }

// Load through the UI module's classloader: no notebook dependency on plugin implementation classes.
val provider = IndexableSetContributor.EP_NAME.extensionList.single {
    it.javaClass.name == "org.limepepper.lang.wikitext.search.WtManagedWtbotIndexableSetContributor"
}
val probeType = provider.javaClass.classLoader.loadClass(
    "org.limepepper.lang.wikitext.search.WtManagedReplaceProbe"
)
val probeCompanion = probeType.getField("Companion").get(null)

fun probeCall(target: Any, name: String, vararg arguments: Any): Any? {
    val method = target.javaClass.methods.single { it.name == name && it.parameterCount == arguments.size }
    return try { method.invoke(target, *arguments) }
    catch (failure: InvocationTargetException) { throw failure.targetException }
}
println("Project: ${probeProject.name}; target: $targetPath")


In [ ]:
val probe = requireNotNull(probeCall(probeCompanion, "prepare", probeProject, targetPath, findText, replacementText))
val before = requireNotNull(probeCall(probe, "getBefore"))
val originalText = probeCall(before, "getBackend") as String
val backupDirectory = Files.createTempDirectory("wikisource-replace-probe-")
Files.writeString(backupDirectory.resolve("original.wikitext"), originalText)
Files.writeString(backupDirectory.resolve("target.txt"), targetPath)
println("Original text backed up to: $backupDirectory")
println(probeCall(before, "report"))
println("Expected content is stored on the probe as getExpected().")


## Run the real action

Run the next cell, inspect the one-page scope and the two strings in the dialog, and click **Replace All**. Finish any confirmation and wait for completion. Then close the dialog and run the comparison cell.


In [ ]:
probeCall(probe, "openReplaceDialog")
println("Complete Replace All in the IDE, then run the next cell.")


In [ ]:
val afterDialog = requireNotNull(probeCall(probe, "capture"))
println(probeCall(probe, "report", afterDialog))


Interpret the first divergence:

- Managed document unchanged: investigate matching, usage selection, or writability.
- Managed document changed and still unsaved, backend unchanged: replacement ran but save did not complete.
- Backend changed, managed VFS unchanged: managed cache/refresh issue.
- Backend and managed views changed, legacy views unchanged: bridge/legacy cache issue.
- Legacy VFS changed, legacy document unchanged: document reload issue.

Snapshots and reports do not intentionally refresh or save either document. Original strings remain available in `before` and the backup file.

## Restore

Close the replacement dialog first. Run this even if the comparisons failed. It restores the original backend text using its current revision token, then refreshes and reloads the two target documents. It refuses to overwrite unexpected concurrent content or document edits.


In [ ]:
val restored = requireNotNull(probeCall(probe, "restore"))
println(probeCall(restored, "report"))
println("Original text restored; backup remains at $backupDirectory")


## Optional: compare with direct native replacement

To isolate replacement from dialog/search enumeration, run this directly after preparation, skipping the dialog cells. If you already used the dialog, restore first. This invokes IntelliJ's native usage-replacement method and explicitly saves the managed document, bypassing dialog/search enumeration. It restores the original text in `finally`.

If this works but the dialog route fails, the adapter can save and the discrepancy is earlier in the native dialog/batch path.


In [ ]:
try {
    println("Native usages processed: ${probeCall(probe, "replaceDirectly")}")
    val afterDirect = requireNotNull(probeCall(probe, "capture"))
    println(probeCall(probe, "report", afterDirect))
} finally {
    val restoredAfterDirect = requireNotNull(probeCall(probe, "restore"))
    println(probeCall(restoredAfterDirect, "report"))
}
